# SAS Test Notebook

This notebook is updated to match the current `broncho_mas` SAS code.

I made this one more for checking the engineering side of the agent, so it does not try to be a polished teacher demo. The point is to show:
- what payload/state we send into SAS
- which skill is selected
- what each skill record says internally
- whether the renewed behaviours still pass simple regression checks

The main behaviours checked here are:
- landmark teaching only fires when it should
- panic/lost messages select support instead of normal QA
- route guidance from the carina names the anatomical path, not only the final target
- RB5 to RB6 does not tell the student to do RB4/RB5 again
- off-topic questions get a short redirect and then return to the bronchoscopy task


## 1. Setup

In [ ]:
import os
import sys
import importlib
import pkgutil
from pathlib import Path

project_root = Path.cwd()
src_path = project_root / "src"

if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Simple popup key input if needed
if not os.environ.get("OPENAI_API_KEY"):
    try:
        import tkinter as tk
        from tkinter import simpledialog
        root = tk.Tk()
        root.withdraw()
        key = simpledialog.askstring("OpenAI API Key", "Enter OpenAI API key:", show="*")
        try:
            root.destroy()
        except Exception:
            pass
        if key:
            os.environ["OPENAI_API_KEY"] = key.strip()
    except Exception:
        pass

os.environ.setdefault("BRONCHO_PROVIDER", "openai")
os.environ.setdefault("BRONCHO_MODEL", "gpt-5.4-nano-2026-03-17")
os.environ.setdefault("BRONCHO_REPORT_USE_LLM", "1")

provider = os.environ.get("BRONCHO_PROVIDER", "").strip().lower()
USE_LLM = provider == "openai" and bool(os.environ.get("OPENAI_API_KEY"))

from broncho_mas.sas.manager import SASManager
from broncho_mas.sas.skill_policy import SkillPolicyLoader, SkillPolicyCompiler
from broncho_mas.sas.skill_registry import SKILL_REGISTRY

# Robust shared-module discovery
import broncho_mas.shared as shared_pkg

shared_module_names = [m.name for m in pkgutil.iter_modules(shared_pkg.__path__)]
print("shared modules =", shared_module_names)

bronchoscopy_knowledge = None
for name in [
    "bronchoscopy_knowledge",
    "knowledge",
    "anatomy",
    "bronchoscopy",
]:
    if name in shared_module_names:
        bronchoscopy_knowledge = importlib.import_module(f"broncho_mas.shared.{name}")
        print(f"loaded knowledge module: broncho_mas.shared.{name}")
        break

print("project_root =", project_root)
print("src_path     =", src_path)
print("provider     =", provider)
print("USE_LLM      =", USE_LLM)
print("skills       =", list(SKILL_REGISTRY.keys()))

## 2. Registry and policy sanity checks

In [ ]:

for name, spec in SKILL_REGISTRY.items():
    print("=" * 80)
    print("name               :", name)
    print("descriptor         :", getattr(spec, "descriptor", None))
    print("category           :", getattr(spec, "category", None))
    print("produces_utterance :", getattr(spec, "produces_utterance", None))
    print("backend            :", getattr(getattr(spec, "backend", None), "__name__", repr(getattr(spec, "backend", None))))


In [ ]:

loader = SkillPolicyLoader()
compiler = SkillPolicyCompiler(loader=loader)

runtime_policy = compiler.compile_for_stage("runtime_guidance")
report_policy = compiler.compile_for_stage("reporting")

print("RUNTIME POLICY")
print("-" * 80)
print(runtime_policy[:1800])

print("\n" + "=" * 80 + "\n")

print("REPORT POLICY")
print("-" * 80)
print(report_policy[:1200])


## 3. Knowledge layer quick check

In [ ]:
import broncho_mas.shared.curriculum as curriculum

print("curriculum module =", curriculum.__name__)
print()

for name in dir(curriculum):
    if name.isupper():
        value = getattr(curriculum, name)
        if isinstance(value, (list, tuple, dict, set)):
            print(f"{name}: {type(value).__name__}, size={len(value)}")
        else:
            print(f"{name}: {type(value).__name__}")

## 4. Manager initialization

In [ ]:

sas = SASManager()
print("sas =", type(sas).__name__)
print("AIRWAY_VISIT_ORDER length =", len(getattr(sas, "AIRWAY_VISIT_ORDER", [])))
print("AIRWAY_VISIT_ORDER =", getattr(sas, "AIRWAY_VISIT_ORDER", []))


## 5. Payload helpers

These helpers keep the test cells shorter. `make_payload()` builds one synthetic robot/student state, and the small record helpers pull out the internal skill data so we can see why SAS chose a response.


In [ ]:
def make_payload(**overrides):
    payload = {
        "iteration": 1,
        "mode": "Automatic",
        "visualization_mode": "Raw",
        "robot_joints": [0.0, 0.0, 0.0],
        "m_jointsVelRel": [0.0, 0.0, 0.0],
        "bounding_boxes": [],
        "current_airway": "RB2",
        "target_airway": "RB3",
        "requested_next_airway": "RB3",
        "reached_regions": ["RB1", "RB2"],
        "phase": "navigation",
        "need_llm": True,
        "llm_reason": "Need short live coaching for next navigation step.",
        "soft_prompt": "Short coaching sentence for bronchoscopy navigation.",
        "previous_msgs": "Hold center. Keep the lumen visible.",
        "student_question": "",
        "is_centered": True,
        "is_stable": True,
        "is_target_visible": False,
        "drift_detected": False,
        "wall_contact_risk": False,
        "need_recenter": False,
        "duration_seconds": 300,
        "backtrack_ratio": 0.10,
        "meta": {
            "source": "sas_current_code_test_notebook",
            "note": "synthetic payload for SAS tests",
        },
    }
    payload.update(overrides)
    return payload


def get_skill_record(result: dict, skill_name: str):
    skill_records = result.get("skill_records", []) if isinstance(result, dict) else []
    return next((r for r in skill_records if r.get("skill") == skill_name), None)


def get_skill_data(result: dict, skill_name: str) -> dict:
    record = get_skill_record(result, skill_name) or {}
    return record.get("data") or {}


def get_landmark_record(result: dict):
    return get_skill_record(result, "landmark_teaching_skill")


def get_question_mode(result: dict) -> str:
    data = get_skill_data(result, "qa_skill")
    return data.get("question_mode") or (data.get("frame") or {}).get("question_mode") or "none"


def get_support_mode(result: dict) -> str:
    data = get_skill_data(result, "support_skill")
    return data.get("support_mode") or "none"


def print_skill_table(result: dict):
    print("  skill_records:")
    for record in result.get("skill_records", []):
        data = record.get("data") or {}
        short_data = {
            key: data.get(key)
            for key in ["support_mode", "question_mode", "frame_mode", "realized", "qa_allowed"]
            if key in data
        }
        print(
            "    -",
            record.get("skill"),
            "active=", record.get("active"),
            "priority=", record.get("priority"),
            "data=", short_data,
        )


def print_result_summary(title: str, result: dict):
    raw = result.get("raw", {}) if isinstance(result, dict) else {}
    normalized = raw.get("normalized_state", {}) if isinstance(raw, dict) else {}
    landmark_record = get_landmark_record(result)

    print(title)
    print("  selected_skill       :", result.get("selected_skill"))
    print("  ui_text              :", result.get("ui_text"))
    print("  support_mode         :", get_support_mode(result))
    print("  question_mode        :", get_question_mode(result))
    print("  guidance_text        :", result.get("guidance_utterance_full"))
    print("  qa_text              :", result.get("qa_utterance_full"))
    print("  support_text         :", result.get("support_utterance_full"))
    print("  landmark_record      :", landmark_record)
    print("  skills_used          :", raw.get("skills_used"))
    print("  need_llm             :", normalized.get("need_llm", raw.get("need_llm")))
    print("  llm_reason           :", normalized.get("llm_reason", raw.get("llm_reason")))
    print("  used_fallback_backend:", raw.get("used_fallback_backend", result.get("used_fallback_backend")))
    print("  normalized current   :", normalized.get("current_airway"))
    print("  normalized target    :", normalized.get("target_airway"))
    print_skill_table(result)


def print_landmark_focus(name: str, result: dict):
    landmark_record = get_landmark_record(result)
    print("\nCASE:", name)
    if landmark_record is None:
        print("  landmark_record      : None")
        return
    print("  landmark active      :", landmark_record.get("active"))
    print("  landmark id          :", landmark_record.get("data", {}).get("landmark_id"))
    print("  display name         :", landmark_record.get("data", {}).get("display_name"))
    print("  first_arrival        :", landmark_record.get("data", {}).get("first_arrival"))
    print("  recognition cues     :", landmark_record.get("data", {}).get("recognition_cues"))
    print("  utterance            :", landmark_record.get("utterance"))



## 6. Smoke tests for renewed SAS behaviour

This cell runs synthetic cases through `sas.step()`. I am checking both the user-visible sentence and the internal skill record, because the engineering question is not only "what did it say?" but also "why did this skill win?"


In [ ]:
from copy import deepcopy

sas_cases = [
    ("sas_basic", make_payload()),
    (
        "sas_landmark_first_arrival_case",
        make_payload(
            iteration=11,
            current_airway="CARINA",
            target_airway="RB1",
            requested_next_airway="RB1",
            reached_regions=["CARINA"],
            validated_landmark="CARINA",
            previous_landmark="TRACHEA",
            first_time_landmark=True,
            just_reached=True,
            student_question="",
            wall_contact_risk=False,
            need_recenter=False,
            drift_detected=False,
            is_target_visible=True,
            previous_msgs="Hold steady.",
            duration_seconds=180,
            backtrack_ratio=0.05,
        ),
    ),
    (
        "sas_landmark_repeat_suppressed_case",
        make_payload(
            iteration=12,
            current_airway="CARINA",
            target_airway="RB1",
            requested_next_airway="RB1",
            reached_regions=["CARINA"],
            validated_landmark="CARINA",
            previous_landmark="CARINA",
            first_time_landmark=False,
            just_reached=False,
            landmark_teaching_history=["L1_CARINA"],
            student_question="",
            wall_contact_risk=False,
            need_recenter=False,
            drift_detected=False,
            is_target_visible=True,
            previous_msgs="Carina - reset anchor.",
        ),
    ),
    (
        "sas_landmark_safety_suppressed_case",
        make_payload(
            iteration=13,
            current_airway="CARINA",
            target_airway="RB1",
            requested_next_airway="RB1",
            reached_regions=["CARINA"],
            validated_landmark="CARINA",
            previous_landmark="TRACHEA",
            first_time_landmark=True,
            just_reached=True,
            student_question="",
            wall_contact_risk=True,
            need_recenter=True,
            drift_detected=True,
            is_centered=False,
            is_stable=False,
            previous_msgs="Hold steady.",
        ),
    ),
    (
        "sas_landmark_alias_case",
        make_payload(
            iteration=14,
            current_airway="BI",
            target_airway="RB4",
            requested_next_airway="RB4",
            reached_regions=["CARINA", "RUL", "BI"],
            validated_landmark="BI",
            previous_landmark="RUL",
            first_time_landmark=True,
            just_reached=True,
            student_question="",
            wall_contact_risk=False,
            need_recenter=False,
            drift_detected=False,
            is_target_visible=True,
            previous_msgs="Good. Keep going.",
        ),
    ),
    (
        "sas_student_panic_support_case",
        make_payload(
            iteration=20,
            current_airway="CARINA",
            target_airway="RB1",
            requested_next_airway="RB1",
            reached_regions=["CARINA"],
            student_question="I'm panicking",
            landmark_teaching_history=["L1_CARINA"],
            previous_msgs="",
        ),
    ),
    (
        "sas_student_lost_support_case",
        make_payload(
            iteration=21,
            current_airway="RMB",
            target_airway="RB6",
            requested_next_airway="RB6",
            reached_regions=["CARINA", "RMB"],
            student_question="I'm lost",
            landmark_teaching_history=["L1_CARINA"],
            previous_msgs="",
        ),
    ),
    (
        "sas_carina_to_rb6_route_case",
        make_payload(
            iteration=22,
            current_airway="CARINA",
            target_airway="RB6",
            requested_next_airway="RB6",
            reached_regions=["CARINA"],
            landmark_teaching_history=["L1_CARINA"],
            previous_msgs="",
        ),
    ),
    (
        "sas_rb5_to_rb6_no_repeat_middle_lobe_case",
        make_payload(
            iteration=23,
            current_airway="RB5",
            target_airway="RB6",
            requested_next_airway="RB6",
            reached_regions=["CARINA", "RMB", "RML", "RB4", "RB5"],
            previous_msgs="Great, since you've just reached RB5.",
            landmark_teaching_history=["L1_CARINA"],
        ),
    ),
    (
        "sas_safety_case",
        make_payload(
            iteration=2,
            current_airway="RB6",
            target_airway="RB7",
            requested_next_airway="RB7",
            reached_regions=["RB1", "RB2", "RB3", "RB4", "RB5", "RB6"],
            previous_msgs="You are close to the bifurcation.",
            student_question="Can we talk about the weather",
            wall_contact_risk=True,
            need_recenter=True,
            drift_detected=True,
            is_centered=False,
            is_stable=False,
            m_jointsVelRel=[-0.3, 0.0, 0.0],
            duration_seconds=410,
            backtrack_ratio=0.22,
        ),
    ),
    (
        "sas_target_visible_case",
        make_payload(
            iteration=3,
            current_airway="RB4",
            target_airway="RB5",
            requested_next_airway="RB5",
            reached_regions=["RB1", "RB2", "RB3", "RB4"],
            is_target_visible=True,
            m_jointsVelRel=[0.1, -0.1, 0.0],
            student_question="I can already see the next opening.",
            duration_seconds=360,
            backtrack_ratio=0.12,
        ),
    ),
]

sas_results = []
for name, payload in sas_cases:
    print("\n" + "=" * 80)
    print("SAS CASE:", name)

    result = sas.step(deepcopy(payload))
    sas_results.append((name, result))

    raw_payload = result.get("raw", {}).get("normalized_state", {}).get("raw_payload", {})
    state = result.get("raw", {}).get("normalized_state", {})

    print_result_summary("sas summary", result)
    print("  raw_payload.m_jointsVelRel:", raw_payload.get("m_jointsVelRel"))
    print("  state.m_jointsVelRel      :", state.get("m_jointsVelRel"))
    print()



## 7. Landmark-teaching-focused inspection

In [ ]:

for name, result in sas_results:
    if "landmark" in name:
        print_landmark_focus(name, result)


## 8. Targeted QA debug inspection

This cell is specifically for student questions. The important thing here is the `question_mode` inside the QA skill record. It shows whether the router treated the student sentence as navigation-related, visual/observation-related, emotional support, or off-task.


In [ ]:
qa_cases = [
    ("question_next_step", make_payload(student_question="Where do I go next?")),
    ("question_observation", make_payload(
        current_airway="RB4",
        target_airway="RB5",
        requested_next_airway="RB5",
        reached_regions=["RB1", "RB2", "RB3", "RB4"],
        is_target_visible=True,
        student_question="I can already see the next opening.",
        m_jointsVelRel=[0.1, -0.1, 0.0],
    )),
    ("question_panic_support", make_payload(
        current_airway="CARINA",
        target_airway="RB1",
        requested_next_airway="RB1",
        reached_regions=["CARINA"],
        student_question="I'm panicking",
        landmark_teaching_history=["L1_CARINA"],
    )),
    ("question_lost_support", make_payload(
        current_airway="RMB",
        target_airway="RB6",
        requested_next_airway="RB6",
        reached_regions=["CARINA", "RMB"],
        student_question="I'm lost",
        landmark_teaching_history=["L1_CARINA"],
    )),
    ("question_off_topic_redirect", make_payload(
        current_airway="CARINA",
        target_airway="RB6",
        requested_next_airway="RB6",
        reached_regions=["CARINA"],
        student_question="Can you talk about hockey?",
        landmark_teaching_history=["L1_CARINA"],
    )),
]

qa_results = []
for name, payload in qa_cases:
    res = sas.step(payload)
    qa_results.append((name, res))
    print("\n" + "-" * 80)
    print(name)
    print("selected_skill =", res["selected_skill"])
    print("support_mode   =", get_support_mode(res))
    print("question_mode  =", get_question_mode(res))
    print("ui_text        =", res["ui_text"])
    print("guidance_text  =", res.get("guidance_utterance_full"))
    print("support_text   =", res.get("support_utterance_full"))
    print("qa_text        =", res.get("qa_utterance_full"))
    print("used_fallback_backend =", res["raw"].get("used_fallback_backend"))



## 9. Reporting path test

In [ ]:

report_payload = sas.generate_session_report(
    payload={
        "current_airway": "RB4",
        "target_airway": "RB5",
        "reached_regions": ["RB1", "RB2", "RB3", "RB4"],
        "student_questions": 1,
        "duration_seconds": 420,
        "backtrack_ratio": 0.18,
    },
    session_metrics={
        "duration_seconds": 420,
        "backtrack_ratio": 0.18,
        "student_questions": 1,
    },
    sp_score=0.82,
    teach_line="Overall teaching focus: continue building a systematic segment-by-segment bronchoscopy technique."
)

print("report keys =", sorted(report_payload.keys()))
print("-" * 80)
print(report_payload.get("report_text", "")[:2500])


## 10. Legacy prompt compatibility

In [ ]:

legacy_prompt = '''
CURRENT_SITUATION: Current airway: RB2
Target airway: RB3
reached_regions: ["RB1", "RB2"]
m_jointsVelRel: [0.1, 0.0, 0.0]
Need LLM: true

PREVIOUS_MSGS: Hold steady. Keep the lumen centered.

STUDENT_QUESTION: Where do I go next?
'''.strip()

legacy_result = sas.run(legacy_prompt)
print_result_summary("legacy sas summary", legacy_result)


## 11. Assertions

These are intentionally plain. If one fails, it points to the behaviour that changed: registry, landmark teaching, support routing, anatomical route text, off-topic redirect, or legacy compatibility.


In [ ]:
assert len(SKILL_REGISTRY) >= 6, "Registry looks too small."
assert "landmark_teaching_skill" in SKILL_REGISTRY, "Registry missing landmark_teaching_skill."
assert "qa_skill" in SKILL_REGISTRY, "Registry missing qa_skill."
assert all(isinstance(r, tuple) and isinstance(r[1], dict) for r in sas_results), "Unexpected result structure."

results = dict(sas_results)
qa_dict = dict(qa_results)

for name, result in sas_results:
    assert "selected_skill" in result, f"{name}: missing selected_skill"
    assert "ui_text" in result, f"{name}: missing ui_text"
    assert "raw" in result, f"{name}: missing raw block"
    assert "skill_records" in result, f"{name}: missing skill_records"

# landmark first-arrival: should at least create a landmark record and activate
landmark_case = results["sas_landmark_first_arrival_case"]
landmark_record = get_landmark_record(landmark_case)
assert landmark_record is not None, "First-arrival case missing landmark_teaching_skill record"
assert landmark_record.get("active") is True, "Landmark teaching should activate on first arrival"
assert landmark_record.get("utterance"), "Landmark teaching utterance is empty for first-arrival case"

# repeat case: should stay inactive if repeat suppression works
repeat_case = results["sas_landmark_repeat_suppressed_case"]
repeat_record = get_landmark_record(repeat_case)
assert repeat_record is not None, "Repeat case missing landmark_teaching_skill record"
assert repeat_record.get("active") in {False, None}, "Repeat case should not actively fire teaching"

# safety case: should stay inactive under safety pressure
safety_landmark_case = results["sas_landmark_safety_suppressed_case"]
safety_landmark_record = get_landmark_record(safety_landmark_case)
assert safety_landmark_record is not None, "Safety-suppressed case missing landmark_teaching_skill record"
assert safety_landmark_record.get("active") in {False, None}, "Safety case should not actively fire teaching"

# alias case: should at least create a landmark record and activate
alias_case = results["sas_landmark_alias_case"]
alias_record = get_landmark_record(alias_case)
assert alias_record is not None, "Alias case missing landmark_teaching_skill record"
assert alias_record.get("active") is True, "Alias case should activate landmark teaching"

# panic/lost should be handled by support, not generic QA or guidance
panic_case = results["sas_student_panic_support_case"]
assert panic_case["selected_skill"]["skill"] == "support_skill", "Panic case should select support_skill"
assert get_support_mode(panic_case) == "distress_recovery", "Panic case should expose distress_recovery support mode"
assert "pause" in panic_case.get("ui_text", "").lower(), "Panic support should start with pausing"

lost_case = results["sas_student_lost_support_case"]
assert lost_case["selected_skill"]["skill"] == "support_skill", "Lost case should select support_skill"
assert get_support_mode(lost_case) == "lost_orientation", "Lost case should expose lost_orientation support mode"
assert "carina" in lost_case.get("ui_text", "").lower(), "Lost support should use carina as reset anchor"

# carina to RB6 should name the route, not just say 'move to RB6'
rb6_route_case = results["sas_carina_to_rb6_route_case"]
rb6_route_text = rb6_route_case.get("ui_text", "").lower()
assert rb6_route_case["selected_skill"]["skill"] == "guidance_skill", "RB6 route case should select guidance_skill"
assert "right main bronchus" in rb6_route_text, "RB6 route should mention right main bronchus"
assert "bronchus intermedius" in rb6_route_text, "RB6 route should mention bronchus intermedius"
assert "right lower lobe bronchus" in rb6_route_text, "RB6 route should mention right lower lobe bronchus"
assert "right lower lobe superior segment" in rb6_route_text, "RB6 route should include the RB6 anatomical location"

# if already at RB5, the next instruction should not tell the student to complete RB4/RB5 again
rb5_to_rb6_case = results["sas_rb5_to_rb6_no_repeat_middle_lobe_case"]
rb5_to_rb6_text = rb5_to_rb6_case.get("ui_text", "").lower()
rb5_guidance_frame = get_skill_data(rb5_to_rb6_case, "guidance_skill").get("frame") or {}
rb5_frame_text = str(rb5_guidance_frame.get("next_step") or "").lower()
assert "rb6" in rb5_to_rb6_text, "RB5 to RB6 case should still guide toward RB6"
assert "usually sits high" in rb5_frame_text, "RB6 location cue should be present in the guidance frame"
assert "complete rb4 and rb5" not in rb5_to_rb6_text, "Should not repeat RB4/RB5 after the student reached RB5"
assert "complete rb4 and rb5" not in rb5_frame_text, "Guidance frame should not repeat RB4/RB5 either"

# general safety case still needs usable text
safety_case = results["sas_safety_case"]
assert safety_case.get("ui_text"), "Safety case produced empty ui_text"

# QA cases should still produce selected skill + text
for name, result in qa_results:
    assert result.get("selected_skill"), f"{name}: missing selected_skill"
    assert result.get("ui_text"), f"{name}: empty ui_text"

# off-topic should answer briefly and resume the bronchoscopy task
hockey_case = qa_dict["question_off_topic_redirect"]
hockey_text = hockey_case.get("ui_text", "").lower()
assert hockey_case["selected_skill"]["skill"] == "qa_skill", "Off-topic case should still use qa_skill"
assert get_question_mode(hockey_case) == "off_task_social", "Off-topic case should be routed as off_task_social"
assert "save that for later" in hockey_text, "Off-topic case should redirect clearly"
assert "bronchoscopy" in hockey_text, "Off-topic case should resume the task context"
assert "rb6" in hockey_text, "Off-topic case should preserve the navigation target"

# legacy path should still return a normal SAS response
assert "selected_skill" in legacy_result, "Legacy result missing selected_skill"
assert "ui_text" in legacy_result, "Legacy result missing ui_text"

print("Notebook smoke assertions passed.")




## 12. Notes

Run order:
1. setup
2. registry / policy checks
3. manager init
4. payload helpers
5. smoke tests
6. landmark inspection
7. QA inspection
8. reporting
9. legacy prompt compatibility
10. assertions

This notebook is meant as a fast regression / inspection notebook for the current SAS codebase. It is useful for an engineering reader because the printed output shows both the final sentence and the internal skill records that produced it.
